# 001 — Understanding Code Sandboxes

学习目标：

1. 理解为什么需要代码沙盒
2. 亲眼看到直接执行代码的危险
3. 理解隔离的四个维度：文件系统、网络、进程、资源
4. 了解 Clawith 沙盒的全景
5. 形成"沙盒是一系列约束的组合"的心智模型

---


## 1. 为什么不是"在服务器上直接执行"？

AI Agent 需要执行代码——数据分析、文件操作、网络请求、安装包……  
但如果你让 Agent 直接在宿主机上跑代码，会出现什么情况？

先看一个无害的例子——你让 Agent "看看当前目录"：


In [ ]:
import subprocess, os
result = subprocess.run(["ls", "-la"], capture_output=True, text=True)
print(result.stdout)


看起来很正常。但假如 Agent 理解错了你的意图——或者被 prompt injection 攻击——结果可能是这样的：


In [ ]:
# ⚠️ 危险演示：只打印，不执行
print("Agent 收到指令: rm -rf /important_data")
print("如果直接在宿主机执行，所有数据将被删除。")
print("沙盒的作用：在可能造成损害之前拦住它。")


### Fork Bomb 演示（理论）

最经典的 DoS 攻击——fork bomb：


In [ ]:
# ⚠️ 危险演示：公式展示，不执行
print("Fork bomb 代码（不要在任何地方运行）：")
print("  :(){ :|:& };:")
print()
print("这条命令会无限创建子进程，几秒内耗尽系统 PID 和 CPU。")
print("沙盒通过 RLIMIT_NPROC 限制子进程数量来防御。")


## 2. 隔离的四个维度

我们把 Agent 执行代码想象成一个访客进入你的服务器：

| 维度 | 不隔离 | 隔离后 |
|---|---|---|
| **文件系统** | 可以读写 `/etc/passwd`、`/root/.ssh` | 只能看到自己的工作目录 |
| **网络** | 可以扫描内网、攻击其他服务 | 无法访问任何网络 |
| **进程** | 可以创建上千进程拖垮系统 | 子进程数受限 |
| **资源** | 可以吃掉所有内存和 CPU | 内存、CPU、磁盘都有限额 |

Clawith 的默认沙盒对这**四个维度都有约束**。


## 3. Clawith 沙盒全景

Clawith 的沙盒系统是**插件式架构**，支持多种后端：

```
                    ┌──────────────┐
                    │  agent_tools │  ← Agent 调用入口
                    └──────┬───────┘
                           │
                    ┌──────▼───────┐
                    │  registry.py │  ← 工厂 + 注册表
                    └──────┬───────┘
                           │
              ┌────────────┼────────────┐
              ▼            ▼            ▼
        Subprocess      Docker        E2B / ...
       (bwrap 隔离)   (容器隔离)     (云端隔离)
```

**默认后端**是 `SubprocessBackend`，使用 bubblewrap（bwrap）进行 Linux namespace 隔离。

它的设计哲学是**纵深防御（Defense in Depth）**：


In [ ]:
# Clawith 沙盒的纵深防御层次
layers = [
    "① 静态代码扫描  → 拦截明显危险命令",
    "② 路径安全验证  → 防止目录逃逸",
    "③ bwrap namespace → 隔离文件系统/网络/进程",
    "④ setrlimit     → CPU/内存/进程数限额",
    "⑤ chroot        → 进一步锁住文件系统（root 模式）",
    "⑥ fail closed   → bwrap 不可用时拒绝执行",
]
for i, layer in enumerate(layers, 1):
    print(f"  第{i}层: {layer}")


## 4. 文件结构（真实源码）

本教程参考的 Clawith 源码位置：

```text
backend/app/services/sandbox/
├── __init__.py          # 模块导出
├── base.py              # SandboxBackend Protocol + ExecutionResult
├── config.py            # SandboxType + SandboxConfig
├── registry.py          # 工厂函数 + 注册表
├── local/
│   ├── subprocess_backend.py   # 默认后端（bwrap）
│   └── docker_backend.py       # Docker 后端
├── api/
│   ├── e2b_backend.py
│   ├── judge0_backend.py
│   └── codesandbox_backend.py
└── remote/
    ├── self_hosted_backend.py
    └── aio_sandbox_backend.py
```


## 5. 在这之前，做一个实验

在你自己的终端里试试这个命令，理解 bwrap 能做到什么：


In [ ]:
import shutil, sys

bwrap = shutil.which("bwrap")
if bwrap:
    print(f"bwrap 已安装: {bwrap}")
    import subprocess
    # 在 bwrap 里看看"我"是谁
    r = subprocess.run(
        [bwrap, "--ro-bind", "/", "/", "--", "whoami"],
        capture_output=True, text=True, timeout=5
    )
    print(f"bwrap 内的 whoami: {r.stdout.strip() or r.stderr.strip()}")
else:
    print("bwrap 未安装 —— 本教程后续会演示如何处理这种情况")


---

**小结：** 代码沙盒不是要不要的问题，而是怎么设计层次的问题。  
接下来的 Notebook 会用 Python 逐步构建一个简化版 Clawith 沙盒系统。
